# ECG Arrhythmia Classification — Explained Notebook

This notebook follows the implemented Model A pipeline: load ECG segments, extract features, use the shared split manifest, tune Logistic Regression and Random Forest, and evaluate on held-out samples. Each executable statement has a Python `#` explanation comment above it. Tuning may take several minutes. The released dataset lacks patient identifiers, so this sample-stratified benchmark is not patient-independent clinical validation.

## 1. Imports and project path
The notebook imports the same repository APIs used by the training scripts.

In [ ]:
# Import Path to locate the project folder portably.
from pathlib import Path
# Import sys to add the project folder to Python's import path.
import sys
# Import NumPy for arrays and fold index operations.
import numpy as np
# Import pandas for data tables and result summaries.
import pandas as pd
# Use the current notebook folder as the workspace root.
workspace_root = Path.cwd()
# Locate the nested project folder containing src/.
project_root = workspace_root / 'DataScience-Project'
# Support running this notebook from inside DataScience-Project too.
if not project_root.exists() and (workspace_root / 'src').exists():
    project_root = workspace_root
# Add the project folder so imports resolve to its src package.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Import the common dataset loader and ECG feature extractor interface.
from src.features import get_dataset_for_models
# Import the saved split manifest loader.
from src.data.split import load_split_manifest
# Import the two Model A classifiers.
from src.models.model_a.baseline_lr import LogisticRegressionBaseline
from src.models.model_a.random_forest import RandomForestModel
# Import the project's ANSI/AAMI metric calculator.
from src.evaluation.metrics import compute_aami_metrics
# Set class letters in the encoded order N=0, S=1, V=2, F=3, Q=4.
class_labels = ['N', 'S', 'V', 'F', 'Q']


## 2. Load data and engineer features

The shared interface normalizes each waveform and extracts 38 HRV, QRS, P/T, waveform-statistics, and spectral features.

In [ ]:
# Load normalized ECG signals, engineered features, integer labels, and metadata.
dataset = get_dataset_for_models()
# Select the feature table used by the classical models.
X_features = dataset['X_features']
# Select the integer-encoded target labels.
y = dataset['y']
# Keep the original class letters for readable summaries.
raw_labels = dataset['raw_labels']
# Show the number of samples and engineered columns.
print('Feature matrix shape:', X_features.shape)
# Show sample counts for each AAMI class.
print(pd.Series(raw_labels).value_counts().reindex(class_labels))
# Display a few rows to inspect feature names and values.
display(X_features.head())


## 3. Load the shared split and make folds
The manifest holds out test samples and assigns five validation folds within training data.

In [ ]:
# Load the shared manifest created by the project's split-generation script.
manifest = load_split_manifest()
# Get source row indexes reserved for training and cross-validation.
train_indices = manifest.loc[manifest['split'] == 'train', 'sample_index'].to_numpy()
# Get source row indexes reserved for final testing.
test_indices = manifest.loc[manifest['split'] == 'test', 'sample_index'].to_numpy()
# Select training predictors from the feature table.
X_train = X_features.iloc[train_indices].copy()
# Select labels corresponding to those training predictors.
y_train = y[train_indices]
# Select held-out predictors without using them in tuning.
X_test = X_features.iloc[test_indices].copy()
# Select the true labels used only for final scoring.
y_test = y[test_indices]
# Verify that no source sample appears in both partitions.
assert set(train_indices).isdisjoint(set(test_indices))
# Report the training and test sample sizes.
print(f'Train: {len(train_indices)}; held-out test: {len(test_indices)}')
# Start the list of fold-training and fold-validation position pairs.
cv_folds = []
# Visit each of the five validation folds in the manifest.
for fold_number in range(5):
    # Find original sample indexes assigned to this validation fold.
    val_indices = manifest.loc[(manifest['split'] == 'train') & (manifest['cv_fold'] == fold_number), 'sample_index'].to_numpy()
    # Map original sample indexes to positions within X_train.
    val_positions = np.flatnonzero(np.isin(train_indices, val_indices))
    # Use remaining training positions to fit this fold's model.
    fit_positions = np.flatnonzero(~np.isin(train_indices, val_indices))
    # Save positions in the format expected by model tuning methods.
    cv_folds.append((fit_positions, val_positions))
# Confirm that all five folds were loaded.
print('Cross-validation folds:', len(cv_folds))


## 4. Tune and score Logistic Regression
The wrapper compares the configured grid using training folds, selects the best mean macro F1, refits on all training rows, then scores the held-out test set.

In [ ]:
# Create the baseline model with the project's iteration limit and random seed.
lr_model = LogisticRegressionBaseline(max_iter=3000, random_state=42)
# Define regularization strengths, class weighting choices, and solver.
lr_grid = {'C': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0], 'class_weight': [None, 'balanced'], 'solver': ['lbfgs']}
# Search fixed folds and choose the setting with best mean macro F1.
lr_best_params, lr_cv_results = lr_model.tune(X_train, y_train, cv_folds, param_grid=lr_grid)
# Fit the selected model pipeline on all training rows.
lr_model.fit(X_train, y_train)
# Predict the held-out test labels.
lr_predictions = lr_model.predict(X_test)
# Calculate class probabilities for each held-out row.
lr_probabilities = lr_model.predict_proba(X_test)
# Calculate AAMI metrics and confusion matrix for the test predictions.
lr_metrics = compute_aami_metrics(y_test, lr_predictions, class_labels=class_labels)
# Display selected hyperparameters and headline test scores.
print('Best LR settings:', lr_best_params)
print(f"LR test accuracy: {lr_metrics['overall_accuracy']:.1%}")
print(f"LR test macro F1: {lr_metrics['macro_f1']:.1%}")


## 5. Tune and score Random Forest
This grid follows the project's training script and compares tree count, depth, split/leaf sizes, feature sampling, and class weights.

In [ ]:
# Create the Random Forest model wrapper with a fixed random seed.
rf_model = RandomForestModel(random_state=42)
# Define the forest hyperparameter combinations used by the project.
rf_grid = {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 15], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2], 'max_features': ['sqrt', 'log2'], 'class_weight': [None, 'balanced']}
# Search the same folds and select parameters by mean macro F1.
rf_best_params, rf_cv_results = rf_model.tune(X_train, y_train, cv_folds, param_grid=rf_grid)
# Refit the chosen pipeline on the complete training partition.
rf_model.fit(X_train, y_train)
# Predict held-out test labels.
rf_predictions = rf_model.predict(X_test)
# Calculate per-class probabilities for the held-out test rows.
rf_probabilities = rf_model.predict_proba(X_test)
# Calculate the same AAMI test metrics as for Logistic Regression.
rf_metrics = compute_aami_metrics(y_test, rf_predictions, class_labels=class_labels)
# Display selected settings and the final held-out test scores.
print('Best RF settings:', rf_best_params)
print(f"RF test accuracy: {rf_metrics['overall_accuracy']:.1%}")
print(f"RF test macro F1: {rf_metrics['macro_f1']:.1%}")


## 6. Compare models and inspect errors
Confusion matrix rows are true classes and columns are predicted classes. Off-diagonal values show misclassifications.

In [ ]:
# Build a comparison table using the same held-out test partition.
comparison = pd.DataFrame([
    # Record Logistic Regression accuracy, macro F1, and macro recall.
    {'Model': 'Logistic Regression', 'Accuracy': lr_metrics['overall_accuracy'], 'Macro F1': lr_metrics['macro_f1'], 'Macro Recall': lr_metrics['macro_sensitivity']},
    # Record Random Forest accuracy, macro F1, and macro recall.
    {'Model': 'Random Forest', 'Accuracy': rf_metrics['overall_accuracy'], 'Macro F1': rf_metrics['macro_f1'], 'Macro Recall': rf_metrics['macro_sensitivity']},
])
# Display model scores as percentages.
display(comparison.style.format({'Accuracy': '{:.1%}', 'Macro F1': '{:.1%}', 'Macro Recall': '{:.1%}'}))
# Label the Random Forest confusion matrix by actual and predicted classes.
rf_confusion = pd.DataFrame(rf_metrics['confusion_matrix'], index=class_labels, columns=class_labels)
# Identify the matrix rows as true classes.
rf_confusion.index.name = 'True class'
# Identify the matrix columns as predicted classes.
rf_confusion.columns.name = 'Predicted class'
# Display the final Random Forest confusion matrix.
display(rf_confusion)
# Arrange class-specific Random Forest metrics in a table.
rf_class_metrics = pd.DataFrame(rf_metrics['class_metrics']).T
# Show each class's support, recall, precision, specificity, and F1.
display(rf_class_metrics[['Support', 'Sensitivity_Recall', 'PPV_Precision', 'Specificity', 'F1_Score']])


## Interpretation and limitations

The project report lists 92% Logistic Regression and 96% Random Forest test accuracy on its 100-sample held-out set. Results may vary if data, dependencies, code, or the saved split changes. Macro F1 weights each class equally. Verified patient and recording IDs are absent, so the split measures sample-level performance and may place samples from one patient in both partitions. Treat these results as a small project benchmark, not clinical validation.